# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [1]:
import os

# ── Local path: adjust if your repo is stored elsewhere ──────────────────────
PROJECT_ROOT = os.path.dirname(os.path.abspath("assignment1.ipynb"))
print("Project root:", PROJECT_ROOT)

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Project root: /root/autodl-tmp/5329/Assignment1_2026-main
PyTorch: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090


In [2]:
# Install Python dependencies (run once per environment)
#import subprocess, sys
#req_file = os.path.join(PROJECT_ROOT, "requirements.txt")
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", req_file, "-q"])
#subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

---
## Section 0 — Environment Setup

Set the project root and install dependencies.

In [3]:
#import sys, os

#if PROJECT_ROOT not in sys.path:
#    sys.path.insert(0, PROJECT_ROOT)

#os.chdir(PROJECT_ROOT)
#print("Working directory:", os.getcwd())

---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists, delete this section before submission.

In [4]:
#from Tools.download import download_mini

#download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [5]:
#from Tools.preproc import preprocess

#preprocess(
#    train_file="_data/squad/train-mini.json",
#    dev_file="_data/squad/dev-v1.1.json",
#    glove_word_file="_data/glove/glove.mini.txt",
#    target_dir="_data",
#    para_limit=400,
#    ques_limit=50,
#)

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [6]:
from TrainTools.train import train

results = train(
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    num_steps       = 60000,
    batch_size      = 32,
    seed            = 42,
    grad_clip       = 5.0,
    early_stop      = 30,

    optimizer_name  = "adam",
    scheduler_name  = "lambda",   # 论文：warmup + 常数 LR
    loss_name       = "qa_nll",

    learning_rate   = 1e-3,
    beta1           = 0.8,        # 论文值
    beta2           = 0.999,      # 论文值
    eps             = 1e-7,       # 论文值
    weight_decay    = 3e-6,
    warmup_steps    = 1000,

    ema_decay       = 0.9999,     # 论文值

    d_model         = 128,
    dropout         = 0.1,        # 论文值
    dropout_char    = 0.05,
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

100%|██████████| 200/200 [00:37<00:00,  5.31it/s]


STEP      200  loss 1531.537131



100%|██████████| 150/150 [00:07<00:00, 19.79it/s]


VALID(train) loss 19.638001  F1 5.761833  EM 0.583333



100%|██████████| 150/150 [00:07<00:00, 20.00it/s]


...

STEP    36400  loss 0.751387



100%|██████████| 150/150 [00:07<00:00, 19.81it/s]


VALID(train) loss 0.164006  F1 95.591533  EM 92.604167



100%|██████████| 150/150 [00:07<00:00, 19.83it/s]


TEST        loss 4.938947  F1 42.409701  EM 32.895833

Learning rate: [0.001]
Early stopping triggered.
Training finished.  Best F1: 42.7112  Best EM: 33.4792
Best F1: 42.7112  |  Best EM: 33.4792


---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [7]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",

    # must match training config
    d_model       = 128,
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

100%|██████████| 1309/1309 [00:32<00:00, 40.21it/s]


TEST  loss 5.020710  F1 41.227555  EM 31.419016
F1: 41.2276  |  EM: 31.4190  |  Loss: 5.020710
